# Notebook 2: Deploy to AWS

This notebook deploys the Strands weather agent to two AWS compute services:

1. **AWS Lambda** — Serverless, pay-per-invocation, no streaming
2. **AWS Fargate** — Containerized, streaming support, high availability

Every code cell is executable from JupyterLab. Each pattern provisions real AWS resources.

> **⚠️ You only need to run ONE pattern.** Pick the one that fits your use case:
>
> | Use case | Run |
> |----------|-----|
> | Short-lived requests, batch jobs, event-driven | **Pattern 1: Lambda** |
> | Interactive apps, streaming, always-on | **Pattern 2: Fargate** |
>
> Skip the other pattern — they deploy separate infrastructure.

> **Prerequisites:** 
> - Complete [Notebook 01](./01_local_agent_to_http_service.ipynb) first (creates `agent_handler.py`, `app.py`, `Dockerfile`)
> - AWS credentials with permissions for Lambda, IAM, ECR, and Bedrock
> - Docker installed (for Fargate pattern only)

In [ ]:
!pip install strands-agents strands-agents-tools fastapi uvicorn boto3

---
## Pattern 1: AWS Lambda (Serverless)

**Best for:** Short-lived agent interactions, batch processing, event-driven workloads.

| Pros | Cons |
|------|------|
| Zero infrastructure management | No response streaming |
| Pay only for execution time | 15-minute max execution |
| Auto-scales to thousands | Cold starts (~2-5s) |

### Architecture

```
Client → Lambda Function → Bedrock API
              ↓
       agent_handler.py
       (custom dependencies layer)
```

### 1.1 Official Strands Lambda Layer

The Strands team publishes a public Lambda layer with the base `strands-agents` package:

```
arn:aws:lambda:{region}:856699698935:layer:strands-agents-py{python_version}-{architecture}:{layer_version}
```

> **Note:** The AWS account ID `856699698935` is the **official Strands Agents publisher account** — not your account. This is a public layer shared for community use.
>
> Reference: [Strands Agents — Deploy to AWS Lambda](https://strandsagents.com/docs/user-guide/deploy/deploy_to_aws_lambda/)

| Component | Options |
|-----------|--------|
| Python Versions | `3.10`, `3.11`, `3.12`, `3.13` |
| Architectures | `x86_64`, `aarch64` |
| Layer Version 2 | strands-agents v1.40.0 |

Since our agent uses `strands-agents-tools` (for `http_request`), we need a **custom layer** that includes it. The cell below builds and deploys one.

### 1.2 Deploy Lambda with Custom Layer

This cell:
1. Installs `strands-agents` + `strands-agents-tools` into a layer zip
2. Creates an IAM role with Bedrock permissions
3. Publishes the layer and creates the Lambda function

> **Note:** This creates real AWS resources. Run the **Cleanup** cell at the end when done.

In [ ]:
import boto3
import json
import zipfile
import subprocess
import os
import time
import shutil

# Configuration
FUNCTION_NAME = "StrandsWeatherAgent"
REGION = "us-east-1"
ROLE_NAME = "StrandsAgentLambdaRole"

iam_client = boto3.client("iam", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]

# --- Step 1: Build custom dependencies layer ---
print("📦 Building custom dependencies layer...")
layer_dir = "/tmp/strands_layer/python"
if os.path.exists("/tmp/strands_layer"):
    shutil.rmtree("/tmp/strands_layer")
os.makedirs(layer_dir)

# Install dependencies into the layer directory
subprocess.run(
    ["pip", "install", "strands-agents", "strands-agents-tools",
     "--target", layer_dir, "--quiet"],
    check=True
)

# Zip the layer
layer_zip = "/tmp/strands_layer.zip"
with zipfile.ZipFile(layer_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk("/tmp/strands_layer"):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, "/tmp/strands_layer")
            zf.write(file_path, arcname)

layer_size_mb = os.path.getsize(layer_zip) / (1024 * 1024)
print(f"✅ Layer built: {layer_size_mb:.1f} MB")

# --- Step 2: Create IAM role ---
print("\n🔐 Creating IAM role...")
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    role_response = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Role for Strands Agent Lambda function",
    )
    role_arn = role_response["Role"]["Arn"]
    print(f"✅ Created role: {ROLE_NAME}")
except iam_client.exceptions.EntityAlreadyExistsException:
    role_arn = iam_client.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print(f"ℹ️  Role exists: {ROLE_NAME}")

# Attach policies
iam_client.attach_role_policy(
    RoleName=ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
)
iam_client.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="BedrockInvokeAccess",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": "*"
        }]
    }),
)
print("✅ Attached Bedrock + CloudWatch permissions")
print("Waiting for IAM propagation...")
time.sleep(10)

# --- Step 3: Publish the layer ---
print("\n📤 Publishing Lambda layer...")

# Layer is too large for direct upload (>70MB limit).
# Upload to S3 first, then reference it.
s3_client = boto3.client("s3", region_name=REGION)
bucket_name = f"strands-agent-layers-{account_id}-{REGION}"
layer_s3_key = "strands-agents-custom-layer.zip"

# Create S3 bucket (ignore if exists)
try:
    if REGION == "us-east-1":
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": REGION}
        )
    print(f"   Created S3 bucket: {bucket_name}")
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"   S3 bucket exists: {bucket_name}")
except Exception as e:
    if "BucketAlreadyOwnedByYou" in str(e) or "BucketAlreadyExists" in str(e):
        print(f"   S3 bucket exists: {bucket_name}")
    else:
        raise

# Upload layer zip to S3
print(f"   Uploading layer to s3://{bucket_name}/{layer_s3_key} ...")
s3_client.upload_file(layer_zip, bucket_name, layer_s3_key)
print("   Upload complete.")

# Publish layer from S3
layer_response = lambda_client.publish_layer_version(
    LayerName="strands-agents-custom",
    Content={"S3Bucket": bucket_name, "S3Key": layer_s3_key},
    CompatibleRuntimes=["python3.12"],
    Description="strands-agents + strands-agents-tools",
)
layer_arn = layer_response["LayerVersionArn"]
print(f"✅ Layer published: {layer_arn}")

# --- Step 4: Package and create the Lambda function ---
print("\n🚀 Creating Lambda function...")
app_zip = "/tmp/agent_lambda.zip"
with zipfile.ZipFile(app_zip, "w") as zf:
    zf.write("agent_handler.py", "agent_handler.py")

with open(app_zip, "rb") as f:
    zip_content = f.read()

try:
    lambda_client.create_function(
        FunctionName=FUNCTION_NAME,
        Runtime="python3.12",
        Role=role_arn,
        Handler="agent_handler.handler",
        Code={"ZipFile": zip_content},
        Timeout=90,
        MemorySize=512,
        Layers=[layer_arn],
    )
    print(f"✅ Created function: {FUNCTION_NAME}")
except lambda_client.exceptions.ResourceConflictException:
    lambda_client.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=zip_content)
    time.sleep(5)
    lambda_client.update_function_configuration(
        FunctionName=FUNCTION_NAME, Layers=[layer_arn]
    )
    print(f"ℹ️  Updated existing function: {FUNCTION_NAME}")

# Wait for function to be active
waiter = lambda_client.get_waiter("function_active_v2")
waiter.wait(FunctionName=FUNCTION_NAME)
print("✅ Lambda function is ready!")

### 1.3 Invoke the Lambda Function

In [ ]:
import boto3
import json

lambda_client = boto3.client("lambda", region_name="us-east-1")

print("Invoking Lambda function (this may take 30-60s on first call)...")
response = lambda_client.invoke(
    FunctionName="StrandsWeatherAgent",
    Payload=json.dumps(
        {"prompt": "What is the weather in Seattle? (latitude: 47.6062, longitude: -122.3321)"}
    ),
)

result = json.loads(response["Payload"].read().decode())

if "errorMessage" in result:
    print(f"❌ Lambda error: {result['errorMessage']}")
else:
    print("✅ Lambda response:")
    print("=" * 60)
    print(result)

### 1.4 Cleanup Lambda Resources

In [ ]:
import boto3

REGION = "us-east-1"
lambda_client = boto3.client("lambda", region_name=REGION)
iam_client = boto3.client("iam", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)
account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]

# Delete Lambda function
try:
    lambda_client.delete_function(FunctionName="StrandsWeatherAgent")
    print("✅ Deleted Lambda function")
except Exception as e:
    print(f"Lambda: {e}")

# Delete custom layer
try:
    versions = lambda_client.list_layer_versions(LayerName="strands-agents-custom")["LayerVersions"]
    for v in versions:
        lambda_client.delete_layer_version(LayerName="strands-agents-custom", VersionNumber=v["Version"])
    print("✅ Deleted Lambda layer")
except Exception as e:
    print(f"Layer: {e}")

# Delete IAM role
try:
    iam_client.detach_role_policy(
        RoleName="StrandsAgentLambdaRole",
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
    )
    iam_client.delete_role_policy(RoleName="StrandsAgentLambdaRole", PolicyName="BedrockInvokeAccess")
    iam_client.delete_role(RoleName="StrandsAgentLambdaRole")
    print("✅ Deleted IAM role")
except Exception as e:
    print(f"IAM: {e}")

# Delete S3 bucket
bucket_name = f"strands-agent-layers-{account_id}-{REGION}"
try:
    objects = s3_client.list_objects_v2(Bucket=bucket_name).get("Contents", [])
    for obj in objects:
        s3_client.delete_object(Bucket=bucket_name, Key=obj["Key"])
    s3_client.delete_bucket(Bucket=bucket_name)
    print("✅ Deleted S3 bucket")
except Exception as e:
    print(f"S3: {e}")

# Remove local temp files
import shutil
for f in ["/tmp/strands_layer.zip", "/tmp/agent_lambda.zip"]:
    if os.path.exists(f):
        os.remove(f)
if os.path.exists("/tmp/strands_layer"):
    shutil.rmtree("/tmp/strands_layer")

print("\n✅ Lambda pattern cleanup complete!")

---
## Pattern 2: AWS Fargate (Containerized)

**Best for:** Interactive applications, streaming responses, long-running agents.

| Pros | Cons |
|------|------|
| Full streaming support | More infrastructure (VPC, ALB, ECS) |
| No execution time limit | Pay for running tasks even when idle |
| No cold starts | |
| High availability | |

### Architecture

```
Client → ALB → Fargate Task(s) → Bedrock API
                    ↓
              FastAPI + uvicorn
              (Docker container)
```

### 2.1 Build and Push Docker Image to ECR

This cell builds the Docker image (from Notebook 01) and pushes it to Amazon ECR.

In [ ]:
import subprocess
import boto3
import base64

REGION = "us-east-1"
REPO_NAME = "strands-weather-agent"

ecr_client = boto3.client("ecr", region_name=REGION)
sts_client = boto3.client("sts", region_name=REGION)

account_id = sts_client.get_caller_identity()["Account"]
ecr_uri = f"{account_id}.dkr.ecr.{REGION}.amazonaws.com"
image_tag = f"{ecr_uri}/{REPO_NAME}:latest"

# Step 1: Create ECR repository
try:
    ecr_client.create_repository(repositoryName=REPO_NAME)
    print(f"✅ Created ECR repository: {REPO_NAME}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"ℹ️  ECR repository already exists: {REPO_NAME}")

# Step 2: Authenticate Docker to ECR
auth = ecr_client.get_authorization_token()
token = auth["authorizationData"][0]["authorizationToken"]
password = base64.b64decode(token).decode().split(":")[1]

login_result = subprocess.run(
    ["docker", "login", "--username", "AWS", "--password-stdin", ecr_uri],
    input=password, capture_output=True, text=True
)
if login_result.returncode == 0:
    print("✅ Docker authenticated to ECR")
else:
    print(f"❌ Docker login failed: {login_result.stderr}")

# Step 3: Build the Docker image
print("\n🐳 Building Docker image...")
build = subprocess.run(
    ["docker", "build", "-t", image_tag, "."],
    capture_output=True, text=True
)
if build.returncode == 0:
    print("✅ Image built successfully")
else:
    print(f"❌ Build failed:\n{build.stderr[-1000:]}")

# Step 4: Push to ECR
print("\n📤 Pushing to ECR...")
push = subprocess.run(
    ["docker", "push", image_tag],
    capture_output=True, text=True
)
if push.returncode == 0:
    print(f"✅ Image pushed: {image_tag}")
else:
    print(f"❌ Push failed:\n{push.stderr[-500:]}")

print(f"\n📋 Image URI (use this for ECS task definition):")
print(f"   {image_tag}")

### 2.2 Create ECS Cluster and Fargate Service

This cell creates the ECS cluster and runs a Fargate task with the image we just pushed.

> **Note:** For a production setup with ALB and auto-scaling, use AWS CDK or CloudFormation. This cell demonstrates the core ECS/Fargate provisioning.

In [ ]:
import boto3
import json
import time

REGION = "us-east-1"
CLUSTER_NAME = "strands-agent-cluster"
TASK_FAMILY = "strands-weather-agent"
SERVICE_NAME = "strands-weather-service"

ecs_client = boto3.client("ecs", region_name=REGION)
iam_client = boto3.client("iam", region_name=REGION)
sts_client = boto3.client("sts", region_name=REGION)

account_id = sts_client.get_caller_identity()["Account"]
image_uri = f"{account_id}.dkr.ecr.{REGION}.amazonaws.com/strands-weather-agent:latest"

# Step 1: Create ECS cluster
ecs_client.create_cluster(clusterName=CLUSTER_NAME)
print(f"✅ ECS cluster: {CLUSTER_NAME}")

# Step 2: Create task execution role
TASK_ROLE_NAME = "StrandsAgentECSTaskRole"
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "ecs-tasks.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    role_resp = iam_client.create_role(
        RoleName=TASK_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
    )
    task_role_arn = role_resp["Role"]["Arn"]
    print(f"✅ Created task role: {TASK_ROLE_NAME}")
except iam_client.exceptions.EntityAlreadyExistsException:
    task_role_arn = iam_client.get_role(RoleName=TASK_ROLE_NAME)["Role"]["Arn"]
    print(f"ℹ️  Task role exists: {TASK_ROLE_NAME}")

# Attach Bedrock + ECR + CloudWatch permissions
iam_client.attach_role_policy(
    RoleName=TASK_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AmazonECSTaskExecutionRolePolicy"
)
iam_client.put_role_policy(
    RoleName=TASK_ROLE_NAME,
    PolicyName="BedrockAccess",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": "*"
        }]
    }),
)
print("✅ Attached permissions")
time.sleep(10)

# Step 3: Register task definition
ecs_client.register_task_definition(
    family=TASK_FAMILY,
    networkMode="awsvpc",
    requiresCompatibilities=["FARGATE"],
    cpu="256",
    memory="512",
    executionRoleArn=task_role_arn,
    taskRoleArn=task_role_arn,
    containerDefinitions=[{
        "name": "weather-agent",
        "image": image_uri,
        "portMappings": [{"containerPort": 8000, "protocol": "tcp"}],
        "essential": True,
        "environment": [{"name": "AWS_DEFAULT_REGION", "value": REGION}],
    }],
)
print(f"✅ Task definition registered: {TASK_FAMILY}")
print(f"\n📋 Image: {image_uri}")
print(f"📋 Cluster: {CLUSTER_NAME}")
print(f"📋 Task: {TASK_FAMILY}")
print("\n✅ Fargate infrastructure is ready!")
print("\nTo run a task, use the AWS Console or:")
print(f"  aws ecs run-task --cluster {CLUSTER_NAME} --task-definition {TASK_FAMILY} --launch-type FARGATE --network-configuration ...")

### 2.3 Run Fargate Task and Test

Run a Fargate task with a public IP (using the default VPC) and invoke the agent.

In [ ]:
import boto3
import time
import requests

REGION = "us-east-1"
CLUSTER_NAME = "strands-agent-cluster"
TASK_FAMILY = "strands-weather-agent"

ecs_client = boto3.client("ecs", region_name=REGION)
ec2_client = boto3.client("ec2", region_name=REGION)

# Get default VPC and a public subnet
vpcs = ec2_client.describe_vpcs(Filters=[{"Name": "isDefault", "Values": ["true"]}])
vpc_id = vpcs["Vpcs"][0]["VpcId"]

subnets = ec2_client.describe_subnets(
    Filters=[{"Name": "vpc-id", "Values": [vpc_id]}, {"Name": "map-public-ip-on-launch", "Values": ["true"]}]
)
subnet_id = subnets["Subnets"][0]["SubnetId"]

# Create a security group that allows inbound on port 8000
try:
    sg_response = ec2_client.create_security_group(
        GroupName="strands-agent-fargate-sg",
        Description="Allow inbound 8000 for Strands agent",
        VpcId=vpc_id,
    )
    sg_id = sg_response["GroupId"]
    ec2_client.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[{"IpProtocol": "tcp", "FromPort": 8000, "ToPort": 8000, "IpRanges": [{"CidrIp": "0.0.0.0/0"}]}]
    )
    print(f"✅ Created security group: {sg_id}")
except ec2_client.exceptions.ClientError as e:
    if "Duplicate" in str(e):
        sgs = ec2_client.describe_security_groups(Filters=[{"Name": "group-name", "Values": ["strands-agent-fargate-sg"]}])
        sg_id = sgs["SecurityGroups"][0]["GroupId"]
        print(f"ℹ️  Security group exists: {sg_id}")
    else:
        raise

# Run the task with a public IP
print("\n🚀 Running Fargate task...")
run_response = ecs_client.run_task(
    cluster=CLUSTER_NAME,
    taskDefinition=TASK_FAMILY,
    launchType="FARGATE",
    networkConfiguration={
        "awsvpcConfiguration": {
            "subnets": [subnet_id],
            "securityGroups": [sg_id],
            "assignPublicIp": "ENABLED",
        }
    },
)

task_arn = run_response["tasks"][0]["taskArn"]
print(f"   Task ARN: {task_arn}")

# Wait for task to be running
print("   Waiting for task to start...")
waiter = ecs_client.get_waiter("tasks_running")
waiter.wait(cluster=CLUSTER_NAME, tasks=[task_arn], WaiterConfig={"Delay": 10, "MaxAttempts": 30})
print("   Task is RUNNING")

# Get the public IP
task_desc = ecs_client.describe_tasks(cluster=CLUSTER_NAME, tasks=[task_arn])
eni_id = None
for attachment in task_desc["tasks"][0]["attachments"]:
    for detail in attachment.get("details", []):
        if detail["name"] == "networkInterfaceId":
            eni_id = detail["value"]

eni_desc = ec2_client.describe_network_interfaces(NetworkInterfaceIds=[eni_id])
public_ip = eni_desc["NetworkInterfaces"][0]["Association"]["PublicIp"]
print(f"   Public IP: {public_ip}")

# Wait for the app to be ready
print("   Waiting for app to start...")
for i in range(20):
    time.sleep(5)
    try:
        resp = requests.get(f"http://{public_ip}:8000/health", timeout=3)
        if resp.status_code == 200:
            print("   ✅ App is healthy!")
            break
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
        print(f"   Waiting... ({i+1}/20)")
else:
    print("   ❌ App did not become healthy in time")

# Invoke the agent
print("\n📨 Invoking weather agent on Fargate...")
response = requests.post(
    f"http://{public_ip}:8000/weather",
    json={"prompt": "What is the weather in Seattle? (latitude: 47.6062, longitude: -122.3321)"},
    timeout=120,
)

if response.status_code == 200:
    print("✅ Fargate agent response:")
    print("=" * 60)
    print(response.text)
else:
    print(f"❌ Error {response.status_code}: {response.text}")

### 2.3 Cleanup Fargate Resources

In [ ]:
import boto3

REGION = "us-east-1"
ecr_client = boto3.client("ecr", region_name=REGION)
ecs_client = boto3.client("ecs", region_name=REGION)
iam_client = boto3.client("iam", region_name=REGION)
ec2_client = boto3.client("ec2", region_name=REGION)

# Stop running tasks
try:
    tasks = ecs_client.list_tasks(cluster="strands-agent-cluster")["taskArns"]
    for task in tasks:
        ecs_client.stop_task(cluster="strands-agent-cluster", task=task)
    if tasks:
        print(f"✅ Stopped {len(tasks)} running task(s)")
        # Wait for tasks to stop and release ENI/SG
        print("   Waiting for tasks to stop...")
        waiter = ecs_client.get_waiter("tasks_stopped")
        waiter.wait(cluster="strands-agent-cluster", tasks=tasks, WaiterConfig={"Delay": 5, "MaxAttempts": 12})
        print("   Tasks stopped.")
except Exception as e:
    print(f"Tasks: {e}")

# Delete ECR repository
try:
    ecr_client.delete_repository(repositoryName="strands-weather-agent", force=True)
    print("✅ Deleted ECR repository")
except Exception as e:
    print(f"ECR: {e}")

# Delete ECS task definition and cluster
try:
    task_defs = ecs_client.list_task_definitions(familyPrefix="strands-weather-agent")["taskDefinitionArns"]
    for td in task_defs:
        ecs_client.deregister_task_definition(taskDefinition=td)
    ecs_client.delete_cluster(cluster="strands-agent-cluster")
    print("✅ Deleted ECS cluster and task definitions")
except Exception as e:
    print(f"ECS: {e}")

# Delete security group (retry with delay for ENI detachment)
import time
try:
    sgs = ec2_client.describe_security_groups(Filters=[{"Name": "group-name", "Values": ["strands-agent-fargate-sg"]}])
    for sg in sgs["SecurityGroups"]:
        for attempt in range(5):
            try:
                ec2_client.delete_security_group(GroupId=sg["GroupId"])
                print("✅ Deleted security group")
                break
            except ec2_client.exceptions.ClientError as e:
                if "DependencyViolation" in str(e) and attempt < 4:
                    print(f"   SG still in use, retrying in 10s... ({attempt+1}/5)")
                    time.sleep(10)
                else:
                    raise
except Exception as e:
    print(f"SG: {e}")

# Delete IAM role
try:
    iam_client.detach_role_policy(
        RoleName="StrandsAgentECSTaskRole",
        PolicyArn="arn:aws:iam::aws:policy/service-role/AmazonECSTaskExecutionRolePolicy"
    )
    iam_client.delete_role_policy(RoleName="StrandsAgentECSTaskRole", PolicyName="BedrockAccess")
    iam_client.delete_role(RoleName="StrandsAgentECSTaskRole")
    print("✅ Deleted IAM role")
except Exception as e:
    print(f"IAM: {e}")

# Remove local Docker images
import subprocess
account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
image_tag = f"{account_id}.dkr.ecr.{REGION}.amazonaws.com/strands-weather-agent:latest"
subprocess.run(["docker", "rmi", "-f", image_tag], capture_output=True)
subprocess.run(["docker", "rmi", "-f", "strands-weather-agent"], capture_output=True)
print("✅ Removed local Docker images")

print("\n✅ Fargate pattern cleanup complete!")

---
## Comparison Summary

| Criteria | Lambda | Fargate |
|----------|--------|--------|
| **Streaming** | ❌ | ✅ |
| **Cold start** | ~2-5s | None |
| **Max duration** | 15 min | Unlimited |
| **Infrastructure** | Minimal | VPC + ECS |
| **Scaling** | Per-request | Task-based |
| **Cost model** | Per-invocation | Per-hour |
| **Best for** | Batch, event-driven | Interactive apps |

### Decision Flowchart

```
Need streaming? ──No──► Lambda
       │
      Yes
       │
Need full infra control? ──Yes──► EKS or EC2
       │
      No
       ↓
    Fargate
```

> **See also:** For managed runtime with session isolation, see [Bedrock AgentCore](https://strandsagents.com/docs/user-guide/deploy/deploy_to_bedrock_agentcore/).

## Summary

In this notebook, you deployed the same Strands weather agent to two AWS services:

| Pattern | What was provisioned |
|---------|---------------------|
| **Lambda** | IAM role + custom layer + Lambda function |
| **Fargate** | ECR repo + Docker image + ECS cluster + task definition + running task |

> **See also:** [Bedrock AgentCore](https://strandsagents.com/docs/user-guide/deploy/deploy_to_bedrock_agentcore/) for managed runtime with session isolation.

**Next:** In [Notebook 03](./03_production_best_practices.ipynb), we'll add production hardening: security, performance, observability, and cost optimization.